# 第16章　敵対的頑健性 ― AIをだます入力と、その守り

**『医療診断支援AIの社会実装（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## だます仕組みを、式と最小コードで

In [ ]:
# FGSM: 誤らせる方向へ画素をepsilonだけ動かす
model.zero_grad(set_to_none=True)   # 既存のパラメータ勾配を消してから攻撃を計算する
# 下のbackward()で再び勾配がたまるため、後続の学習更新前にもzero_grad()を呼ぶ。
x.requires_grad_(True)
loss = criterion(model(x), y)
loss.backward()
x_adv = (x + eps * x.grad.sign()).clamp(0, 1)   # 摂動を加えてクリップ
# model(x_adv) は、見た目ほぼ同じなのに誤分類しうる

## 「たぶん頑健」から「証明つき頑健」へ ― 認証付き頑健性

In [ ]:
import numpy as np
import torch

from scipy.stats import norm   # 認証半径 R = σ Φ^{-1}(p_A) の Φ^{-1} に使う

def certify(x, n0=100, n=1000, sigma=0.25):
    def vote(k):
        v = np.zeros(num_classes)
        for _ in range(k):
            v[model(x + sigma*torch.randn_like(x)).argmax()] += 1
        return v
    # クラスの選択と、下側信頼限界の推定には、独立した標本を使う。
    # 同じ投票で「最多クラスを選び」かつ「その得票率の下限を取る」と、
    # 選択のバイアスが乗り、指定した信頼水準を保てなくなるおそれがある。
    cls = int(vote(n0).argmax())               # ①少数回でクラスだけ決める
    votes = vote(n)                            # ②独立にn回引き直す
    p_a = lower_conf_bound(votes[cls], n)     # 得票率の下側信頼限界
    if not np.isfinite(p_a) or p_a <= 0.5:
        return None, 0.0                     # 棄権：クラスを提示しない
    R = sigma * norm.ppf(p_a)
    return cls, R                            # 認証できたクラスと半径